In [ ]:
pip install pandas openpyxl

In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
from openai import OpenAI

import json
import re
from pathlib import Path

from openpyxl import Workbook
QUESTION_MODEL = "openai/gpt-5.2-chat"
ANSWER_MODEL = "google/gemini-3-flash-preview"
EVALUATION_MODEL = "openai/gpt-5.2-chat"
MAX_OUTPUT_TOKENS = 10000
CLARIFICATION_MAX_TOKENS = 10000
TEMPERATURE = 0
TOP_P = 1

INPUT_EXCEL = "requirements.xlsx"

REQ_ID_COL = "RequirementID"
FUNC_REQ_COL = "FunctionalRequirements"
USE_CASE_COL = "UseCaseScenarios"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY environment variable not set.")

In [ ]:
DIAGRAM_ROOT = "generated_diagrams"

OUTPUT_WORKBOOK = "evaluation_results.xlsx"

CLARIFICATION_CACHE_FILE = "clarification_cache.json"

MAX_CLARIFICATION_ROUNDS = 1

PROMPT_MAPPING = {
    "Single Prompt_Zero Shot": "Simple_0.puml",
    "Single Prompt_One Shot": "Simple_1.puml",
    "Single Prompt_Few Shot": "Simple_Few.puml",
    "CoT_Zero Shot": "CoT_0.puml",
    "CoT_Few Shot": "CoT_Few.puml"
}

In [ ]:
OUTPUT_COLUMNS = [
    "RequirementID",
    "PromptType",
    "DiagramFile",
    "EvaluationStatus",
    "ClarificationRounds",

    "DelimiterScore",
    "StartEndScore",

    "IncorrectActionNodes",
    "TotalActionNodes",

    "IncorrectDecisionNodes",
    "TotalDecisionNodes",

    "SemanticallyIncorrectActionNodes",
    "SemanticallyIncorrectDecisionNodes",

    "TotalFlowsInGoldStandard",
    "SemanticallyIncorrectFlows",

    "TotalActionsInScenario",
    "ActionsCovered",

    "TotalDecisionsInScenario",
    "DecisionsCovered",

    "TotalGeneratedNodes",
    "RedundantNodes",

    "HallucinatedNodes",

    "TotalImplicitNodesGenerated",

    "SwimlaneIdentificationScore"
]

In [ ]:
def load_clarification_cache():

    if not os.path.exists(CLARIFICATION_CACHE_FILE):
        return {}

    with open(CLARIFICATION_CACHE_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def save_clarification_cache(cache):

    with open(CLARIFICATION_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2)

In [ ]:
def ask_llm(prompt, model_name=None):
    print(f"\nCalling model: {model_name}")
    if model_name is None:
        model_name = QUESTION_MODEL

    import time
    if(model_name == ANSWER_MODEL):
        max_tokens = CLARIFICATION_MAX_TOKENS
    else:
        max_tokens = MAX_OUTPUT_TOKENS
        
    for attempt in range(3):
    
        try:
    
            response = client.chat.completions.create(
                model=model_name,
                temperature=TEMPERATURE,
                max_tokens=max_tokens,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
    
            break
    
        except Exception as e:
    
            print(f"Attempt {attempt+1} failed: {e}")
    
            if attempt == 2:
                raise
    
            time.sleep(10)


    if not response.choices:
        print("\nFULL RESPONSE:")
        print(response)
        
        raise Exception(
                f"WARNING: No choices returned."
        )

    message = response.choices[0].message

    if message is None:
        raise Exception(
            f"No message returned\n{response}"
        )

    if message.content is None:

        print(
            "WARNING: Model returned no content."
        )

        return None

    with open(
        "last_response.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(message.content)
    
    return message.content.strip()

In [ ]:
def answer_clarification_questions(
    functional_requirement,
    use_case_scenario,
    questions
):

    prompt = f"""
{CLARIFICATION_ANSWER_PROMPT}

Functional Requirement:
{functional_requirement}

Use Case Scenario:
{use_case_scenario}

Clarification Questions:
{questions}
"""

    response = ask_llm(
        prompt,
        model_name=ANSWER_MODEL
    )

    if response is None:
        return "Not specified in requirement."

    return response.strip()

In [ ]:
UNDERSTANDING_PROMPT = """
You are an expert software engineer specialized in UML modeling and requirements analysis.

Your task is ONLY to understand the requirement.

You will be given:

1. Functional Requirement
2. Use Case Scenario
3. Previously asked questions and human answers.

IMPORTANT:

- Never repeat a question that has already been answered.
- Treat previous answers as authoritative.
- Ask ONLY new questions that remain unresolved.
- If previous answers resolve all ambiguities, return exactly:

NO_CLARIFICATION_REQUIRED

Rules:

1. Ask at most 5 questions.
2. Ask only genuinely necessary questions.
3. Do not restate answered questions.
4. Do not summarize previous answers.
"""

In [ ]:
CLARIFICATION_ANSWER_PROMPT = """
You are an expert software requirements analyst.

You will receive:

1. Functional Requirement
2. Use Case Scenario
3. Clarification Questions

Answer the questions using only the information that can be reasonably inferred from the requirement and use case scenario.

Rules:

- Answer every question.
- Be concise.
- If information is unavailable, state:
  "Not specified in requirement."
- Return plain text answers only.
"""

In [ ]:
def get_clarification_history(
    requirement_id,
    functional_requirement,
    use_case_scenario
):

    cache = load_clarification_cache()

    if requirement_id in cache:
        print(f"Using cached clarification for {requirement_id}")

        return cache[requirement_id]

    history = []

    for round_num in range(MAX_CLARIFICATION_ROUNDS):

        prompt = f"""
{UNDERSTANDING_PROMPT}

Requirement ID:
{requirement_id}

Functional Requirement:
{functional_requirement}

Use Case Scenario:
{use_case_scenario}

Previous Clarifications:
{json.dumps(history, indent=2)}
"""

        response = ask_llm(prompt,QUESTION_MODEL)
        
        if response is None:
        
            print(
                "No clarification response returned."
            )
        
            response = "NO_CLARIFICATION_REQUIRED"
        
        if "NO_CLARIFICATION_REQUIRED" in response:
        
            cache[requirement_id] = {
                "completed": True,
                "rounds_used": len(history),
                "clarifications": history
            }
        
            save_clarification_cache(cache)
        
            return cache[requirement_id]
    
        print("\n")
        print("=" * 80)
        print(f"Requirement: {requirement_id}")
        print(f"Clarification Round {round_num+1}")
        print("=" * 80)
        print(response)
        
        answer = answer_clarification_questions(
            functional_requirement,
            use_case_scenario,
            response
        )
        
        print("\nLLM Clarification Answers:")
        print(answer)
        
        history.append({
            "questions": response,
            "answers": answer
        })
    
    cache[requirement_id] = {
        "completed": True,
        "rounds_used": len(history),
        "clarifications": history
    }
    
    save_clarification_cache(cache)
    
    return cache[requirement_id]

In [ ]:
METRIC_DEFINITIONS = r"""
You are an expert software engineer specialized in UML modeling and requirements analysis.

Assess the activity diagram using the following metric definitions.

====================================================
1. SYNTACTIC CORRECTNESS
====================================================

A. DelimiterScore

Score:
1 = @startuml and @enduml exist and are correctly placed
0 = otherwise

----------------------------------------------------

B. StartEndScore

Score:
1 = proper start and end nodes are used
0 = otherwise

----------------------------------------------------

C. IncorrectActionNodes

Action nodes must follow PlantUML activity syntax:

:Action Text;

Count an action node as syntactically incorrect if:

- missing leading ':'
- missing trailing ';'
- malformed syntax
- invalid activity notation

TotalActionNodes =
Total action nodes present in generated diagram

----------------------------------------------------

D. IncorrectDecisionNodes

Decision nodes must:

- use valid PlantUML if syntax
- contain valid branches
- contain proper outcome labels
  such as yes/no, true/false, approve/reject

Count as incorrect if:

- malformed if statement
- missing branch labels
- invalid decision structure

TotalDecisionNodes =
Total decision nodes present in generated diagram

====================================================
2. SEMANTIC CORRECTNESS
====================================================

First construct an INTERNAL GOLD STANDARD activity model using:

- Functional Requirement
- Use Case Scenario
- Clarification Answers

----------------------------------------------------

A. SemanticallyIncorrectActionNodes

Count action nodes that:

- contradict requirement
- represent wrong behavior
- perform incorrect actor responsibilities
- misrepresent intended action

----------------------------------------------------

B. SemanticallyIncorrectDecisionNodes

Count decision nodes that:

- do not exist in requirement
- use incorrect conditions
- misrepresent branching logic

----------------------------------------------------

C. SemanticallyIncorrectFlows

A flow is a directed control-flow transition.

Count flows that:

- connect wrong nodes
- reverse intended sequence
- omit required transitions
- introduce invalid transitions

TotalFlowsInGoldStandard =
Number of flows in internally constructed gold standard.

====================================================
3. COMPLETENESS
====================================================

TotalActionsInScenario =
Number of distinct actions in gold standard.

ActionsCovered =
Number of gold-standard actions correctly represented.

----------------------------------------------------

TotalDecisionsInScenario =
Number of decisions in gold standard.

DecisionsCovered =
Number of decisions correctly represented.

====================================================
4. REDUNDANCY
====================================================

RedundantNodes =
Nodes that:

1. add no new behavior
2. duplicate existing functionality
3. can be removed without changing meaning
4. are not required implicit nodes

====================================================
5. HALLUCINATION
====================================================

HallucinatedNodes =
Nodes that:

1. are not present in requirement
2. are not present in gold standard
3. are not reasonably inferable
4. introduce unjustified functionality
5. introduce unjustified actors
6. introduce unjustified decisions
7. introduce unjustified processing

====================================================
6. IMPLICIT NODES
====================================================

TotalImplicitNodesGenerated =

Count nodes that:

- are not explicitly written
- are reasonably necessary
- improve process continuity
- represent valid intermediate behavior

Do NOT count redundant or hallucinated nodes.

====================================================
7. SWIMLANE IDENTIFICATION
====================================================

SwimlaneIdentificationScore

1.0
= all swimlanes correctly identified
= no missing swimlanes
= no extra swimlanes

0.5
= some swimlanes correct
OR
= redundant swimlanes exist
OR
= hallucinated swimlanes exist

0
= no swimlanes correctly captured

====================================================
OUTPUT RULES
====================================================

Return ONLY valid JSON.

Do not return explanations.

Do not return markdown.

Do not return reasoning.

Populate every field.

IMPORTANT:

Do not explain your reasoning.

Do not justify scores.

Do not provide examples.

Return ONLY the JSON object.
"""

In [ ]:
EVALUATION_PROMPT_TEMPLATE = """
You are an expert software engineer specialized in UML modeling and requirements analysis.

Use the following as the reference for evaluation:

1. Functional Requirement
2. Use Case Scenario
3. Clarification Answers

Internally infer the intended activity workflow.

Do not output the inferred workflow.

Use it only for scoring.

CRITICAL:

Do not explain.
Do not justify.
Do not provide reasoning.
Do not describe the evaluation process.

Return ONLY a single JSON object.

The entire response must be under 800 tokens.

JSON schema:

{
  "RequirementID":"",
  "PromptType":"",
  "DiagramFile":"",
  "EvaluationStatus":"",
  "ClarificationRounds":0,
  "DelimiterScore":0,
  "StartEndScore":0,
  "IncorrectActionNodes":0,
  "TotalActionNodes":0,
  "IncorrectDecisionNodes":0,
  "TotalDecisionNodes":0,
  "SemanticallyIncorrectActionNodes":0,
  "SemanticallyIncorrectDecisionNodes":0,
  "TotalFlowsInGoldStandard":0,
  "SemanticallyIncorrectFlows":0,
  "TotalActionsInScenario":0,
  "ActionsCovered":0,
  "TotalDecisionsInScenario":0,
  "DecisionsCovered":0,
  "TotalGeneratedNodes":0,
  "RedundantNodes":0,
  "HallucinatedNodes":0,
  "TotalImplicitNodesGenerated":0,
  "SwimlaneIdentificationScore":0
}

EvaluationStatus must be one of:

SUCCESS
FAILED
NEEDS_REVIEW
"""

In [ ]:
def evaluate_diagram(
    requirement_id,
    functional_requirement,
    use_case_scenario,
    clarification_data,
    prompt_type,
    diagram_file,
    plantuml_text
):

    prompt = f"""
{METRIC_DEFINITIONS}

{EVALUATION_PROMPT_TEMPLATE}

Requirement ID:
{requirement_id}

Functional Requirement:
{functional_requirement}

Use Case Scenario:
{use_case_scenario}

Clarifications:
{json.dumps(clarification_data, indent=2)}

Prompt Type:
{prompt_type}

Diagram File:
{diagram_file}

PlantUML Diagram:
{plantuml_text}
"""

    response = ask_llm(prompt,EVALUATION_MODEL)
    
    try:
    
        start = response.find("{")
    
        end = response.rfind("}")
    
        response = response[start:end+1]
    
        result = json.loads(response)
    
    except Exception:
    
        print("INVALID JSON")
    
        with open(
            "invalid_json_response.txt",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(response)
    
        result = {
            "RequirementID": requirement_id,
            "PromptType": prompt_type,
            "DiagramFile": diagram_file,
            "EvaluationStatus": "NEEDS_REVIEW"
        }
    return result

In [ ]:
def create_output_workbook():

    wb = Workbook()

    first_sheet = wb.active
    first_sheet.title = list(PROMPT_MAPPING.keys())[0]

    for col_idx, col_name in enumerate(OUTPUT_COLUMNS, start=1):
        first_sheet.cell(
            row=1,
            column=col_idx,
            value=col_name
        )

    for sheet_name in list(PROMPT_MAPPING.keys())[1:]:

        ws = wb.create_sheet(sheet_name)

        for col_idx, col_name in enumerate(
            OUTPUT_COLUMNS,
            start=1
        ):
            ws.cell(
                row=1,
                column=col_idx,
                value=col_name
            )

    wb.save(OUTPUT_WORKBOOK)

In [ ]:
from openpyxl import load_workbook

def already_evaluated(
    sheet_name,
    requirement_id
):

    if not os.path.exists(OUTPUT_WORKBOOK):
        return False

    wb = load_workbook(
        OUTPUT_WORKBOOK,
        read_only=True
    )

    ws = wb[sheet_name]

    for row in ws.iter_rows(
        min_row=2,
        values_only=True
    ):

        if (
            row[0] == requirement_id
            and row[3] == "SUCCESS"
        ):
            return True

    return False

def append_result(sheet_name, result):

    wb = load_workbook(OUTPUT_WORKBOOK)

    ws = wb[sheet_name]

    next_row = ws.max_row + 1

    for col_idx, col_name in enumerate(
        OUTPUT_COLUMNS,
        start=1
    ):

        ws.cell(
            row=next_row,
            column=col_idx,
            value=result.get(col_name)
        )

    wb.save(OUTPUT_WORKBOOK)

In [ ]:
def read_puml(path):

    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [ ]:
requirements_df = pd.read_excel(INPUT_EXCEL)

create_output_workbook()

for _, row in requirements_df.iterrows():

    requirement_id = str(row[REQ_ID_COL]).strip()

    functional_requirement = str(
        row[FUNC_REQ_COL]
    )

    use_case_scenario = str(
        row[USE_CASE_COL]
    )

    clarification_data = get_clarification_history(
        requirement_id,
        functional_requirement,
        use_case_scenario
    )

    
    requirement_folder = Path(
        DIAGRAM_ROOT
    ) / requirement_id

    for sheet_name, puml_file in PROMPT_MAPPING.items():

        puml_path = requirement_folder / puml_file
        if not puml_path.exists():

            append_result(
                sheet_name,
                {
                    "RequirementID": requirement_id,
                    "PromptType": puml_file.replace(
                        ".puml",
                        ""
                    ),
                    "DiagramFile": puml_file,
                    "EvaluationStatus": "FAILED",
                    "ClarificationRounds":
                        clarification_data["rounds_used"]
                }
            )

            continue

        try:
            if already_evaluated(
                sheet_name,
                requirement_id
            ):

                print(
                    f"Skipping "
                    f"{requirement_id} "
                    f"{puml_file}"
                )

                continue
                
            plantuml_text = read_puml(
                puml_path
            )

            print(
                f"Evaluating {requirement_id} "
                f"{puml_file}"
            )
            
            result = evaluate_diagram(
                requirement_id,
                functional_requirement,
                use_case_scenario,
                clarification_data,
                puml_file.replace(
                    ".puml",
                    ""
                ),
                puml_file,
                plantuml_text
            )

            append_result(
                sheet_name,
                result
            )

            time.sleep(10)

        except Exception as e:

            append_result(
                sheet_name,
                {
                    "RequirementID": requirement_id,
                    "PromptType": puml_file.replace(
                        ".puml",
                        ""
                    ),
                    "DiagramFile": puml_file,
                    "EvaluationStatus": "FAILED",
                    "ClarificationRounds":
                        clarification_data["rounds_used"]
                }
            )

            print(
                f"Error evaluating "
                f"{requirement_id} "
                f"{puml_file}: {e}"
            )

print(
    f"Finished. Results saved to "
    f"{OUTPUT_WORKBOOK}"
)